In [1]:
import pandas as pd
tickets = pd.DataFrame({
    "ticket_id": [1, 2, 3, 4, 5, 6],
    "customer_tier": ["basic", "premium", "basic", "enterprise", "premium", "basic"],
    "severity": ["high", "medium", "critical", "high", "low", "critical"],
    "hours_open": [12, 4, 30, 8, 2, 48],
    "status": ["open", "open", "open", "closed", "open", "open"],
})
tickets=tickets.set_index("ticket_id")
tickets

,customer_tier,severity,hours_open,status
ticket_id,,,,
1,basic,high,12,open
2,premium,medium,4,open
3,basic,critical,30,open
4,enterprise,high,8,closed
5,premium,low,2,open
6,basic,critical,48,open


# Business rules:
  * tickets receive highest priority.
  * tickets receive higher priority than basic tickets.
  * Tickets open for more than  hours require escalation.
  * Closed tickets must not be escalated.

In [3]:
tickets["is_open"]=tickets["status"]=="open"
tickets

,customer_tier,severity,hours_open,status,is_open
ticket_id,,,,,
1,basic,high,12,open,True
2,premium,medium,4,open,True
3,basic,critical,30,open,True
4,enterprise,high,8,closed,False
5,premium,low,2,open,True
6,basic,critical,48,open,True


In [5]:
tickets["needs_escalation"]=(tickets["hours_open"]>24) & (tickets["status"]=="open")
tickets

,customer_tier,severity,hours_open,status,is_open,needs_escalation
ticket_id,,,,,,
1,basic,high,12,open,True,False
2,premium,medium,4,open,True,False
3,basic,critical,30,open,True,True
4,enterprise,high,8,closed,False,False
5,premium,low,2,open,True,False
6,basic,critical,48,open,True,True


In [10]:
# Priority Category
severity_priority={
    "critical":1,
    "high":2,
    "medium":3,
    "low":4
}
tier_priority={
    "enterprise":1,
    "premium":2,
    "basic":3
}

categorical_priority_ticket=tickets.copy()

categorical_priority_ticket["severity_priority"]=tickets["severity"].map(severity_priority)
categorical_priority_ticket["tier_priority"]=tickets["customer_tier"].map(tier_priority)

categorical_priority_ticket=categorical_priority_ticket.sort_values(by=["severity_priority","tier_priority"],ascending=[True,True])

categorical_priority_ticket["priority_rank"]=range(1,len(categorical_priority_ticket)+1)
categorical_priority_ticket


,customer_tier,severity,hours_open,status,is_open,needs_escalation,severity_priority,tier_priority,priority_rank
ticket_id,,,,,,,,,
3,basic,critical,30,open,True,True,1,3,1
6,basic,critical,48,open,True,True,1,3,2
4,enterprise,high,8,closed,False,False,2,1,3
1,basic,high,12,open,True,False,2,3,4
2,premium,medium,4,open,True,False,3,2,5
5,premium,low,2,open,True,False,4,2,6


In [12]:
severity_score={
    "critical":40,
    "high":30,
    "medium":20,
    "low":10
}

tier_score={
    "enterprise":15,
    "premium":10,
    "basic":5
}
priority_score_ticket=tickets.copy()

priority_score_ticket["priority_score"]=(
    priority_score_ticket["severity"].map(severity_score) +
    priority_score_ticket["customer_tier"].map(tier_score) +
    priority_score_ticket["needs_escalation"]*20 +
    priority_score_ticket["hours_open"]

)

priority_score_ticket=priority_score_ticket.sort_values("priority_score",ascending=False)
priority_score_ticket



,customer_tier,severity,hours_open,status,is_open,needs_escalation,priority_score
ticket_id,,,,,,,
6,basic,critical,48,open,True,True,113
3,basic,critical,30,open,True,True,95
4,enterprise,high,8,closed,False,False,53
1,basic,high,12,open,True,False,47
2,premium,medium,4,open,True,False,34
5,premium,low,2,open,True,False,22


In [15]:
action_tickets=priority_score_ticket[priority_score_ticket["needs_escalation"] | priority_score_ticket["is_open"]]
action_tickets

,customer_tier,severity,hours_open,status,is_open,needs_escalation,priority_score
ticket_id,,,,,,,
6,basic,critical,48,open,True,True,113
3,basic,critical,30,open,True,True,95
1,basic,high,12,open,True,False,47
2,premium,medium,4,open,True,False,34
5,premium,low,2,open,True,False,22


When multiple rules are applied, the order determines which rule takes precedence.

For example:

1. Assign severity priority.
2. Assign customer tier priority.
3. Apply escalation rules.
4. Exclude closed tickets from escalation.

If the escalation rule is applied before checking ticket status, a closed ticket that has been open for more than 24 hours may be incorrectly marked for escalation.

Therefore, business rules should be applied in a logical order:
- First determine whether the ticket is open.
- Then check escalation conditions.
- Finally apply priority and ranking rules.

In this dataset, the rule "Closed tickets must not be escalated" overrides the rule "Open for more than 24 hours requires escalation."